In [ ]:
pip install google-api-python-client

  Using cached protobuf-6.31.1-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 16.1 MB/s eta 0:00:00a 0:00:01
Using cached protobuf-6.31.1-cp39-abi3-macosx_10_9_universal2.whl (425 kB)
Using cached pyasn1-0.6.1-py3-none-any.whl (83 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [google-api-python-client]api-python-client]
Note: you may need to restart the kernel to use updated packages.


In [20]:
import re
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd

from dotenv import load_dotenv
import os

In [21]:
# openning file with data
df = pd.read_csv('/Users/katemamyan/Documents/GitHub/LEDE_PROGRAM/project_01/archive_data.csv', index_col=False)
df.head()

,link,video_link,title,time,date
0,https://www.1tv.am/hy/video/Հարցազրույց-Սոսի-Թ...,https://www.youtube.com/embed/gkLMht3Vt-k?auto...,Հարցազրույց Սոսի Թաթիկյանի հետ,22:10,"03 Հլս, 2025"
1,https://www.1tv.am/hy/video/Հարցազրույց-Համբիկ...,https://www.youtube.com/embed/yjj8ISwxTwE?auto...,Հարցազրույց Համբիկ Սարաֆյանի հետ,22:10,"02 Հլս, 2025"
2,https://www.1tv.am/hy/video/Հարցազրույց-Կայա-Կ...,https://www.youtube.com/embed/Hd6l_IRhygI?auto...,Հարցազրույց Կայա Կալասի հետ,22:10,"01 Հլս, 2025"
3,https://www.1tv.am/hy/video/Հարցազրույց-Արթուր...,https://www.youtube.com/embed/d5x7uI6cCLI?auto...,Հարցազրույց Արթուր Պողոսյանի հետ,22:10,"30 Հնս, 2025"
4,https://www.1tv.am/hy/video/Հարցազրույց-Էլեն-Հ...,https://www.youtube.com/embed/gpIDqXFZx8o?auto...,Հարցազրույց Էլեն Հոխիկյանի հետ,22:30,"27 Հնս, 2025"


In [7]:
load_dotenv(override=True)
api_key = os.environ['API_KEY']

In [10]:
def get_video_id(youtube_link):
    """Extracts the video ID from a YouTube URL."""
    # Regex to match various YouTube URL formats
    # Handles watch?v=, youtu.be/, embed/, and shorts/
    match = re.search(r"(?:v=|youtu\.be/|embed/|shorts/)([\w-]{11})", youtube_link)
    if match:
        return match.group(1)
    return None

In [13]:
def get_video_stats(video_ids, api_key):
    """
    Fetches statistics (views, likes, comments, shares) for a list of YouTube video IDs.
    """
    youtube = build("youtube", "v3", developerKey=api_key)
    stats_data = {}

    # The API allows up to 50 video IDs per request
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        try:
            request = youtube.videos().list(
                part="statistics",
                id=",".join(batch_ids)
            )
            response = request.execute()

            for item in response.get("items", []):
                video_id = item["id"]
                statistics = item.get("statistics", {})
                stats_data[video_id] = {
                    "views": statistics.get("viewCount", "N/A"),
                    "likes": statistics.get("likeCount", "N/A"),
                    "comments": statistics.get("commentCount", "N/A"),
                    # Shares are not directly exposed as a discrete count in the API for individual videos
                    # You might infer sharing activity from public comments or external tracking if needed
                    "shares": "N/A (Not directly available via API for individual videos)"
                }
        except HttpError as e:
            print(f"An HTTP error {e.resp.status} occurred: {e.content}")
        except Exception as e:
            print(f"An error occurred: {e}")
    return stats_data

if __name__ == "__main__":
    youtube_links = df['video_link']

    video_ids = []
    for link in youtube_links:
        vid_id = get_video_id(link)
        if vid_id:
            video_ids.append(vid_id)
        else:
            print(f"Could not extract video ID from: {link}")

    if video_ids:
        all_video_stats = get_video_stats(video_ids, api_key)

        for link in youtube_links:
            vid_id = get_video_id(link)
            if vid_id and vid_id in all_video_stats:
                stats = all_video_stats[vid_id]
                print(f"\n--- Stats for: {link} (ID: {vid_id}) ---")
                print(f"  Views: {stats['views']}")
                print(f"  Likes: {stats['likes']}")
                print(f"  Comments: {stats['comments']}")
                print(f"  Shares: {stats['shares']}") # Note: Shares often 'N/A'
            elif vid_id:
                print(f"\n--- No stats found for: {link} (ID: {vid_id}). Might be private, deleted, or incorrect ID. ---")
            else:
                print(f"\n--- Skipping link: {link} (Invalid or unextractable ID) ---")
    else:
        print("No valid video IDs found in the provided list.")

Could not extract video ID from: https://www.facebook.com/plugins/video.php?href=https%3A%2F%2Fwww.facebook.com%2Flurer1tv%2Fvideos%2F1285800781771444%2F&show_text=0&width=560&autoplay=1
Could not extract video ID from: https://www.facebook.com/plugins/video.php?href=https%3A%2F%2Fwww.facebook.com%2Flurer.1tv%2Fvideos%2F322899835336041%2F&show_text=0&width=560&autoplay=1
Could not extract video ID from: https://www.facebook.com/plugins/video.php?href=https%3A%2F%2Fwww.facebook.com%2Flurer.1tv%2Fvideos%2F501143377462564%2F&show_text=0&width=560&autoplay=1

--- Stats for: https://www.youtube.com/embed/gkLMht3Vt-k?autoplay=1&autohide=1 (ID: gkLMht3Vt-k) ---
  Views: 33761
  Likes: 410
  Comments: N/A
  Shares: N/A (Not directly available via API for individual videos)

--- Stats for: https://www.youtube.com/embed/yjj8ISwxTwE?autoplay=1&autohide=1 (ID: yjj8ISwxTwE) ---
  Views: 33991
  Likes: 518
  Comments: N/A
  Shares: N/A (Not directly available via API for individual videos)

--- Stat

In [17]:
all_video_stats

{'gkLMht3Vt-k': {'views': '33761',
  'likes': '410',
  'comments': 'N/A',
  'shares': 'N/A (Not directly available via API for individual videos)'},
 'yjj8ISwxTwE': {'views': '33991',
  'likes': '518',
  'comments': 'N/A',
  'shares': 'N/A (Not directly available via API for individual videos)'},
 'Hd6l_IRhygI': {'views': '42727',
  'likes': '888',
  'comments': 'N/A',
  'shares': 'N/A (Not directly available via API for individual videos)'},
 'd5x7uI6cCLI': {'views': '97758',
  'likes': '1384',
  'comments': 'N/A',
  'shares': 'N/A (Not directly available via API for individual videos)'},
 'gpIDqXFZx8o': {'views': '42014',
  'likes': '504',
  'comments': 'N/A',
  'shares': 'N/A (Not directly available via API for individual videos)'},
 'L9QYke6Lavg': {'views': '145988',
  'likes': '3177',
  'comments': 'N/A',
  'shares': 'N/A (Not directly available via API for individual videos)'},
 'YQ3IWIXioi8': {'views': '100581',
  'likes': '1183',
  'comments': 'N/A',
  'shares': 'N/A (Not direc

In [28]:


def get_video_id(youtube_link):
    """Extracts the video ID from a YouTube URL."""
    match = re.search(r"(?:v=|youtu\.be/|embed/|shorts/)([\w-]{11})", youtube_link)
    if match:
        return match.group(1)
    return None

def parse_iso8601_duration(iso_duration):
    """
    Parses an ISO 8601 duration string (e.g., 'PT1H2M3S') into total seconds.
    Returns duration in HH:MM:SS format or total seconds.
    """
    try:
        duration_obj = isodate.parse_duration(iso_duration)
        total_seconds = int(duration_obj.total_seconds())

        hours, remainder = divmod(total_seconds, 3600)
        minutes, seconds = divmod(remainder, 60)

        # Return as HH:MM:SS string
        return f"{hours:02}:{minutes:02}:{seconds:02}"
    except Exception:
        return "N/A" # Handle cases where parsing fails

def get_video_stats(video_ids, api_key):
    """
    Fetches statistics (views, likes, comments) for a list of YouTube video IDs.
    Returns a dictionary mapping video_id to its statistics.
    """
    youtube = build("youtube", "v3", developerKey=api_key)
    stats_data = {}

    # The API allows up to 50 video IDs per request
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        try:
            request = youtube.videos().list(
                part="statistics",
                id=",".join(batch_ids)
            )
            response = request.execute()

            for item in response.get("items", []):
                video_id = item["id"]
                statistics = item.get("statistics", {})
                stats_data[video_id] = {
                    "views": int(statistics.get("viewCount", 0)),  # Convert to int, default to 0
                    "likes": int(statistics.get("likeCount", 0)),  # Convert to int, default to 0
                    "comments": int(statistics.get("commentCount", 0)), # Convert to int, default to 0
                    "shares": "N/A" # Still not directly available via API
                    
                }
        except HttpError as e:
            print(f"An HTTP error {e.resp.status} occurred for batch: {batch_ids}. Error: {e.content}")
        except Exception as e:
            print(f"An unexpected error occurred for batch: {batch_ids}. Error: {e}")
    return stats_data

# --- SEPARATE BLOCK FOR DURATION FETCHING ---
def get_video_stats_with_duration(video_ids, api_key):
    """
    Fetches statistics (views, likes, comments) and contentDetails (duration)
    for a list of YouTube video IDs.
    Returns a dictionary mapping video_id to its statistics including duration.
    """
    youtube = build("youtube", "v3", developerKey=api_key)
    stats_data = {}

    # The API allows up to 50 video IDs per request
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        try:
            request = youtube.videos().list(
                part="statistics,contentDetails", # Requesting both statistics and contentDetails
                id=",".join(batch_ids)
            )
            response = request.execute()

            for item in response.get("items", []):
                video_id = item["id"]
                statistics = item.get("statistics", {})
                content_details = item.get("contentDetails", {}) # Get contentDetails
                
                duration_iso = content_details.get("duration", "N/A")
                duration_formatted = parse_iso8601_duration(duration_iso)

                stats_data[video_id] = {
                    "views": int(statistics.get("viewCount", 0)),
                    "likes": int(statistics.get("likeCount", 0)),
                    "comments": int(statistics.get("commentCount", 0)),
                    "shares": "N/A", # Still not directly available via API
                    "duration": duration_formatted # Add duration
                }
        except HttpError as e:
            print(f"An HTTP error {e.resp.status} occurred for batch: {batch_ids}. Error: {e.content}")
        except Exception as e:
            print(f"An unexpected error occurred for batch: {batch_ids}. Error: {e}")
    return stats_data
# --- END SEPARATE BLOCK ---

if __name__ == "__main__":
    # 1. Extract Video IDs and add as a new column
    df['video_id'] = df['video_link'].apply(get_video_id)

    # Get a list of unique valid video IDs to fetch stats for
    # Filter out None values in case some links didn't yield an ID
    valid_video_ids = [vid for vid in df['video_id'].unique() if vid]

    if valid_video_ids:
        print(f"\n--- Fetching Views, Likes, Comments, AND Duration for {len(valid_video_ids)} unique video IDs ---")
        all_stats_data = get_video_stats_with_duration(valid_video_ids, api_key)

        df_all_stats = df.copy() # Create another copy
        df_all_stats['views'] = df_all_stats['video_id'].map(lambda x: all_stats_data.get(x, {}).get('views', 'N/A'))
        df_all_stats['likes'] = df_all_stats['video_id'].map(lambda x: all_stats_data.get(x, {}).get('likes', 'N/A'))
        df_all_stats['comments'] = df_all_stats['video_id'].map(lambda x: all_stats_data.get(x, {}).get('comments', 'N/A'))
        df_all_stats['shares'] = df_all_stats['video_id'].map(lambda x: all_stats_data.get(x, {}).get('shares', 'N/A'))
        df_all_stats['duration'] = df_all_stats['video_id'].map(lambda x: all_stats_data.get(x, {}).get('duration', 'N/A'))


        print("\nDataFrame with Views, Likes, Comments, AND Duration:")
        print(df_all_stats)
        print("-" * 30)

    else:
        print("No valid video IDs found in the DataFrame. No stats will be added.")
        df['views'] = 'N/A'
        df['likes'] = 'N/A'
        df['comments'] = 'N/A'
        df['shares'] = 'N/A'
        df['duration'] = 'N/A' # Add duration column even if no stats


    print("\nDataFrame with YouTube Stats:")
    print(df.head())


--- Fetching Views, Likes, Comments, AND Duration for 1262 unique video IDs ---

DataFrame with Views, Likes, Comments, AND Duration:
                                                   link  \
0     https://www.1tv.am/hy/video/Հարցազրույց-Սոսի-Թ...   
1     https://www.1tv.am/hy/video/Հարցազրույց-Համբիկ...   
2     https://www.1tv.am/hy/video/Հարցազրույց-Կայա-Կ...   
3     https://www.1tv.am/hy/video/Հարցազրույց-Արթուր...   
4     https://www.1tv.am/hy/video/Հարցազրույց-Էլեն-Հ...   
...                                                 ...   
1270  https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...   
1271  https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...   
1272  https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...   
1273  https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...   
1274  https://www.1tv.am/hy/video/Հարցազրույց-Արսեն-...   

                                             video_link  \
0     https://www.youtube.com/embed/gkLMht3Vt-k?auto...   
1     https://www.youtube.com/embed/yj

In [29]:
df

,link,video_link,title,time,date,video_id,views,likes,comments,shares
0,https://www.1tv.am/hy/video/Հարցազրույց-Սոսի-Թ...,https://www.youtube.com/embed/gkLMht3Vt-k?auto...,Հարցազրույց Սոսի Թաթիկյանի հետ,22:10,"03 Հլս, 2025",gkLMht3Vt-k,33761,410,0,N/A
1,https://www.1tv.am/hy/video/Հարցազրույց-Համբիկ...,https://www.youtube.com/embed/yjj8ISwxTwE?auto...,Հարցազրույց Համբիկ Սարաֆյանի հետ,22:10,"02 Հլս, 2025",yjj8ISwxTwE,33991,518,0,N/A
2,https://www.1tv.am/hy/video/Հարցազրույց-Կայա-Կ...,https://www.youtube.com/embed/Hd6l_IRhygI?auto...,Հարցազրույց Կայա Կալասի հետ,22:10,"01 Հլս, 2025",Hd6l_IRhygI,42727,888,0,N/A
3,https://www.1tv.am/hy/video/Հարցազրույց-Արթուր...,https://www.youtube.com/embed/d5x7uI6cCLI?auto...,Հարցազրույց Արթուր Պողոսյանի հետ,22:10,"30 Հնս, 2025",d5x7uI6cCLI,97758,1384,0,N/A
4,https://www.1tv.am/hy/video/Հարցազրույց-Էլեն-Հ...,https://www.youtube.com/embed/gpIDqXFZx8o?auto...,Հարցազրույց Էլեն Հոխիկյանի հետ,22:30,"27 Հնս, 2025",gpIDqXFZx8o,42014,504,0,N/A
...,...,...,...,...,...,...,...,...,...,...
1270,https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...,https://www.youtube.com/embed/6g5sFkYDopo?auto...,Հարցազրույց Պետրոս Ղազարյանի հետ. Կարեն Անդրեա...,19:30,"06 Փտր, 2020",6g5sFkYDopo,30397,847,100,N/A
1271,https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...,https://www.youtube.com/embed/g_N7y10q5jE?auto...,Հարցազրույց Պետրոս Ղազարյանի հետ. Էդմոն Մարուքյան,19:30,"05 Փտր, 2020",g_N7y10q5jE,1801,20,14,N/A
1272,https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...,https://www.youtube.com/embed/PLfaKqneN8g?auto...,Հարցազրույց Պետրոս Ղազարյանի հետ. Տիգրան Խաչատ...,19:30,"04 Փտր, 2020",PLfaKqneN8g,2119,34,1,N/A
1273,https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...,https://www.youtube.com/embed/XA3jcUzjQRI?auto...,Հարցազրույց Պետրոս Ղազարյանի հետ. Ռուբեն Ռուբի...,19:30,"03 Փտր, 2020",XA3jcUzjQRI,15580,280,38,N/A


In [27]:
df

,link,video_link,title,time,date,video_id,views,likes,comments,shares
0,https://www.1tv.am/hy/video/Հարցազրույց-Սոսի-Թ...,https://www.youtube.com/embed/gkLMht3Vt-k?auto...,Հարցազրույց Սոսի Թաթիկյանի հետ,22:10,"03 Հլս, 2025",gkLMht3Vt-k,33761,410,0,N/A
1,https://www.1tv.am/hy/video/Հարցազրույց-Համբիկ...,https://www.youtube.com/embed/yjj8ISwxTwE?auto...,Հարցազրույց Համբիկ Սարաֆյանի հետ,22:10,"02 Հլս, 2025",yjj8ISwxTwE,33991,518,0,N/A
2,https://www.1tv.am/hy/video/Հարցազրույց-Կայա-Կ...,https://www.youtube.com/embed/Hd6l_IRhygI?auto...,Հարցազրույց Կայա Կալասի հետ,22:10,"01 Հլս, 2025",Hd6l_IRhygI,42727,888,0,N/A
3,https://www.1tv.am/hy/video/Հարցազրույց-Արթուր...,https://www.youtube.com/embed/d5x7uI6cCLI?auto...,Հարցազրույց Արթուր Պողոսյանի հետ,22:10,"30 Հնս, 2025",d5x7uI6cCLI,97758,1384,0,N/A
4,https://www.1tv.am/hy/video/Հարցազրույց-Էլեն-Հ...,https://www.youtube.com/embed/gpIDqXFZx8o?auto...,Հարցազրույց Էլեն Հոխիկյանի հետ,22:30,"27 Հնս, 2025",gpIDqXFZx8o,42014,504,0,N/A
...,...,...,...,...,...,...,...,...,...,...
1270,https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...,https://www.youtube.com/embed/6g5sFkYDopo?auto...,Հարցազրույց Պետրոս Ղազարյանի հետ. Կարեն Անդրեա...,19:30,"06 Փտր, 2020",6g5sFkYDopo,30397,847,100,N/A
1271,https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...,https://www.youtube.com/embed/g_N7y10q5jE?auto...,Հարցազրույց Պետրոս Ղազարյանի հետ. Էդմոն Մարուքյան,19:30,"05 Փտր, 2020",g_N7y10q5jE,1801,20,14,N/A
1272,https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...,https://www.youtube.com/embed/PLfaKqneN8g?auto...,Հարցազրույց Պետրոս Ղազարյանի հետ. Տիգրան Խաչատ...,19:30,"04 Փտր, 2020",PLfaKqneN8g,2119,34,1,N/A
1273,https://www.1tv.am/hy/video/Հարցազրույց-Պետրոս...,https://www.youtube.com/embed/XA3jcUzjQRI?auto...,Հարցազրույց Պետրոս Ղազարյանի հետ. Ռուբեն Ռուբի...,19:30,"03 Փտր, 2020",XA3jcUzjQRI,15580,280,38,N/A
